# Notebook 3 — SHAP Explainability

**Goal:** Understand *why* the model makes each prediction.

SHAP (SHapley Additive exPlanations) answers:  
*"How much did each feature push this prediction up or down?"

**Run Notebooks 1 and 2 first.**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap
import joblib
import warnings
import os

warnings.filterwarnings('ignore')
# shap.initjs() removed — can crash in some Jupyter/VS Code setups

print('Libraries loaded!')

In [ ]:
# ── Load the model and data ───────────────────────────────────────────────────
model         = joblib.load('../models/best_model.joblib')
feature_names = joblib.load('../models/feature_names.joblib')

X_original = pd.read_csv('../data/processed/X_original.csv')
X_test     = pd.read_csv('../data/processed/X_test.csv')
y_test     = pd.read_csv('../data/processed/y_test.csv').squeeze()

# ── IMPORTANT: align columns to match exactly what the model expects ──────────
X_test = X_test[feature_names]

print(f'Model loaded: {type(model).__name__}')
print(f'Features    : {len(feature_names)}')
print(f'Test rows   : {len(X_test)}')

## 1. Create the SHAP Explainer

For XGBoost we use `TreeExplainer` — it is fast and exact (not an approximation).

In [ ]:
# ── Build the SHAP explainer ──────────────────────────────────────────────────
explainer = shap.TreeExplainer(model)

# Calculate SHAP values for the entire test set
# Each row = one customer, each column = one feature's contribution
shap_values = explainer.shap_values(X_test.values)

print(f'SHAP values shape: {shap_values.shape}')
print(f'  {shap_values.shape[0]} customers x {shap_values.shape[1]} features')

## 2. Summary Plot — What Drives Churn Globally?

- Features at the **top** matter most
- **Red** = customer has a HIGH value for that feature
- **Blue** = customer has a LOW value for that feature
- **Right** side = pushes toward churn | **Left** = pushes toward staying

In [ ]:
plt.figure()
shap.summary_plot(
    shap_values,
    X_test.values,
    feature_names=feature_names,
    show=False,
    max_display=15
)
plt.title('SHAP Summary — Top 15 Features Driving Churn', fontsize=12)
plt.tight_layout()
os.makedirs('../reports/figures', exist_ok=True)
plt.savefig('../reports/figures/13_shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Bar Plot — Feature Importance (Mean |SHAP|)

In [ ]:
plt.figure()
shap.summary_plot(
    shap_values,
    X_test.values,
    feature_names=feature_names,
    plot_type='bar',
    show=False,
    max_display=15
)
plt.title('Mean Feature Importance (SHAP)', fontsize=12)
plt.tight_layout()
plt.savefig('../reports/figures/14_shap_bar.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Waterfall Plot — Explaining One Customer's Prediction

The summary plot explains the whole dataset.  
The waterfall plot explains **one specific customer** — great for your demo.

In [ ]:
# ── Find the highest-risk customer ───────────────────────────────────────────
y_pred_prob   = model.predict_proba(X_test.values)[:, 1]
high_risk_idx = int(np.argmax(y_pred_prob))

print(f'Customer index  : {high_risk_idx}')
print(f'Predicted churn : {y_pred_prob[high_risk_idx]*100:.1f}%')
print(f'Actual outcome  : {"Churned" if y_test.iloc[high_risk_idx] == 1 else "Did not churn"}')

In [ ]:
# ── Waterfall plot for this customer ─────────────────────────────────────────
explanation = shap.Explanation(
    values        = shap_values[high_risk_idx],
    base_values   = float(explainer.expected_value),
    data          = X_test.values[high_risk_idx],
    feature_names = feature_names
)

plt.figure()
shap.waterfall_plot(explanation, show=False, max_display=12)
plt.title(f'Why is Customer #{high_risk_idx} High Risk?', fontsize=11)
plt.tight_layout()
plt.savefig('../reports/figures/15_shap_waterfall.png', dpi=150, bbox_inches='tight')
plt.show()

print()
print('How to read this:')
print('  E[f(x)] = average prediction across all customers (starting point)')
print('  f(x)    = this customer\'s final prediction')
print('  Red bars  = push prediction HIGHER (more likely to churn)')
print('  Blue bars = push prediction LOWER  (less likely to churn)')

## 5. Dependence Plots — How One Feature Affects Churn Risk

In [ ]:
# ── Dependence plot for tenure ────────────────────────────────────────────────
tenure_idx = feature_names.index('tenure')

plt.figure(figsize=(8, 5))
shap.dependence_plot(
    tenure_idx,
    shap_values,
    X_test.values,
    feature_names=feature_names,
    show=False
)
plt.title('How Tenure Affects Churn Risk (SHAP Dependence Plot)', fontsize=11)
plt.tight_layout()
plt.savefig('../reports/figures/16_shap_dependence_tenure.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Dependence plot for MonthlyCharges ────────────────────────────────────────
charges_idx = feature_names.index('MonthlyCharges')

plt.figure(figsize=(8, 5))
shap.dependence_plot(
    charges_idx,
    shap_values,
    X_test.values,
    feature_names=feature_names,
    show=False
)
plt.title('How Monthly Charges Affect Churn Risk', fontsize=11)
plt.tight_layout()
plt.savefig('../reports/figures/17_shap_dependence_charges.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Business Insights from SHAP

In [ ]:
# ── Top features by mean absolute SHAP value ─────────────────────────────────
mean_shap = pd.DataFrame({
    'Feature'  : feature_names,
    'Mean SHAP': np.abs(shap_values).mean(axis=0)
}).sort_values('Mean SHAP', ascending=False).head(10)

print('Top 10 features driving churn predictions:')
print(mean_shap.to_string(index=False))

print()
print('Business insights:')
print('  Month-to-month contract → Offer discounts for annual contracts')
print('  High MonthlyCharges     → Review pricing for high-spend customers')
print('  Low tenure              → Improve onboarding for new customers')
print('  No TechSupport          → Promote TechSupport as a retention tool')

## 7. Save Everything for the Streamlit App

In [ ]:
# ── Save SHAP explainer and values ────────────────────────────────────────────
joblib.dump(explainer,   '../models/shap_explainer.joblib')
joblib.dump(shap_values, '../models/shap_values.joblib')

print('Saved:')
print('  models/shap_explainer.joblib')
print('  models/shap_values.joblib')
print()
print('All 3 notebooks complete!')
print('Launch the app with: streamlit run app/streamlit_app.py')